In [ ]:
# [notebook_constants]

USE_KAGGLE_SANDBOX = True
KAGGLE_SANDBOX_KWARGS = {
    'dataset_data_sources': ['greynearing/camels-us-531-era'],
    'enable_gpu': True,
    'enable_internet': False,
    'machine_shape': 'NvidiaTeslaT4Highmem',
}
C_PUCT = 2.5

# CAMELS-US ERA — PUR (18-Fold HUC-02) Minimal Guidance

## Objective
Maximize the **out-of-sample Median Nash-Sutcliffe Efficiency (NSE)** across all **531 CAMELS-US basins** under the **18-Fold Leave-One-HUC02-Region-Out Prediction in Ungauged Regions (PUR)** cross-validation protocol on the evaluation period (`1999-10-01` to `2003-09-30`, 4 water years, `1,461` days).

## Data & Split Protocol
- **Spatial Regime (18-Fold HUC-02 PUR Cross-Validation)**:
  - `CV_FOLDS` (loaded from `pur_18fold_huc02_splits.json`) partitions the 531 basins by USGS 2-digit Hydrologic Unit Code (`HUC_02`, 18 regions).
  - In each fold `k`, an **entire HUC-02 region** (`fold['test_basin_mask']`) is held out as completely ungauged (`fold['qobs_fold_train']` is `NaN` on that region), and the model is trained on the remaining 17 HUC-02 regions (`fold['train_basin_mask']`) during `2003-10-01` to `2008-09-30`.
  - Predict out-of-sample discharge for `fold['test_idx']` and assemble the full `Q_PRED` array of shape `(531, 3653)` across all 18 regional folds.
- **Strict Newman Temporal Containment (`1998-10-01` to `2008-09-30`, `3,653` days)**:
  - `1998-10-01` to `1999-09-30` (`365` days): 1-year meteorological spin-up prior to evaluation (`QOBS_TRAIN = NaN`).
  - `1999-10-01` to `2003-09-30` (`1,461` days, `ERA_EVAL_MASK`): Out-of-sample ERA hill-climb evaluation period (`QOBS_TRAIN = NaN`).
  - `2003-10-01` to `2008-09-30` (`1,827` days, `ERA_TRAIN_MASK`): ERA model weights training period on each fold's 17 training regions.
- **Inputs Available**:
  - `DYNAMIC_INPUTS`: shape `(531, 3653, 15)` (`float32`) — 5 daily meteorological variables (`prcp`, `srad`, `tmax`, `tmin`, `vp`) across 3 forcing products (`Daymet`, `Maurer`, `NLDAS`).
  - `AUX_FORCINGS`: shape `(531, 3653, 6)` (`float32`) — daily `dayl` and `swe` across `Daymet`, `Maurer`, `NLDAS`.
  - `STATIC_ATTRIBUTES`: shape `(531, 27)` (`float32`) — 27 catchment climate, topography, soil, geology, and vegetation attributes (plus `BASIN_AREAS`, `LAT`, `LON`, `HUC_02`).
- **Required Output**:
  - The mutable cell must produce `Q_PRED` as a `float32` NumPy array of shape `(531, 3653)` containing daily streamflow predictions (`mm/day`).

## Modeling & Exploration Scope
You are free to explore any machine learning architecture, sequence length, feature representation, normalization strategy, loss function, regularization, optimizer schedule, or multi-model ensembling strategy, provided all predictions strictly respect the causal temporal masks and spatial fold boundaries.

In [ ]:
import os
import time
import math
import json
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

_SEARCH_DIRS = [
    Path("/kaggle/input/camels-us-531-era"),
    Path("/kaggle/input/datasets/greynearing/camels-us-531-era"),
    Path("/tmp/kaggle_camels_era_compliant"),
    Path("/tmp/unified_benchmark"),
    Path(__file__).resolve().parent.parent / "splits" if "__file__" in globals() else Path("../splits"),
    Path("../splits"),
    Path("./splits"),
]

def _find_file(filename: str) -> Path:
    for d in _SEARCH_DIRS:
        candidate = d / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {filename} in any of {_SEARCH_DIRS}")

_train_npz = np.load(_find_file("camels_era_train.npz"), allow_pickle=True)
_basins_json = json.loads(_find_file("basins_531.json").read_text(encoding="utf-8"))

BASINS = _train_npz["basins"].astype(str)
_canonical_basins = [str(b) for b in _basins_json]
assert list(BASINS) == _canonical_basins, "BASINS order mismatch against canonical basins_531.json"

# Strictly enforce containment to the Newman training window (1998-10-01 to 2008-09-30, 3653 days)
_raw_dates = np.array([str(d) for d in _train_npz["dates"]])
_newman_window = (_raw_dates >= "1998-10-01") & (_raw_dates <= "2008-09-30")
DATES = _raw_dates[_newman_window]
assert len(DATES) == 3653 and DATES[0] == "1998-10-01" and DATES[-1] == "2008-09-30"

DYNAMIC_INPUTS = _train_npz["dynamic_inputs"][:, _newman_window, :].astype(np.float32)
DYNAMIC_FEATURE_NAMES = [str(x) for x in _train_npz["dynamic_feature_names"]]
AUX_FORCINGS = _train_npz["aux_forcings"][:, _newman_window, :].astype(np.float32)
AUX_FEATURE_NAMES = [str(x) for x in _train_npz["aux_feature_names"]]
STATIC_ATTRIBUTES = _train_npz["static_attributes"].astype(np.float32)
STATIC_ATTRIBUTE_NAMES = [str(x) for x in _train_npz["static_attribute_names"]]

ERA_TRAIN_MASK = (DATES >= "2003-10-01") & (DATES <= "2008-09-30")
ERA_EVAL_MASK = (DATES >= "1999-10-01") & (DATES <= "2003-09-30")
assert int(ERA_TRAIN_MASK.sum()) == 1827, f"Expected 1827 training days, got {int(ERA_TRAIN_MASK.sum())}"
assert int(ERA_EVAL_MASK.sum()) == 1461, f"Expected 1461 eval days, got {int(ERA_EVAL_MASK.sum())}"

QOBS_TRAIN = np.where(
    ERA_TRAIN_MASK[None, :],
    _train_npz["qobs_train"][:, _newman_window].astype(np.float32),
    np.nan,
).astype(np.float32)

BASIN_AREAS = _train_npz["basin_areas"].astype(np.float32)
LAT = _train_npz["lat"].astype(np.float32)
LON = _train_npz["lon"].astype(np.float32)

SPATIAL_REGIME = "pur_18fold"
_splits_json = json.loads(_find_file("pur_18fold_huc02_splits.json").read_text(encoding="utf-8"))
CV_SPLITS = [v for _, v in sorted(_splits_json["folds"].items(), key=lambda kv: int(kv[1]["fold_index"]))]
assert len(CV_SPLITS) == 18, f"Expected 18 PUR folds, got {len(CV_SPLITS)}"

_basin_to_idx = {str(b): i for i, b in enumerate(BASINS)}
CV_FOLDS = []
_covered_test = np.zeros(len(BASINS), dtype=bool)
for _fold in CV_SPLITS:
    _fid = int(_fold["fold_index"])
    _test_idx = np.array([_basin_to_idx[str(b)] for b in _fold["test_basins"]], dtype=np.int64)
    _train_idx = np.array([_basin_to_idx[str(b)] for b in _fold["train_basins"]], dtype=np.int64)
    _test_mask = np.zeros(len(BASINS), dtype=bool)
    _test_mask[_test_idx] = True
    _train_mask = np.zeros(len(BASINS), dtype=bool)
    _train_mask[_train_idx] = True
    assert not np.any(_test_mask & _train_mask), f"Train/test overlap in PUR fold {_fid}"
    assert not np.any(_covered_test & _test_mask), f"Duplicate test basins in PUR fold {_fid}"
    _covered_test |= _test_mask
    CV_FOLDS.append({
        "fold_id": _fid,
        "held_out_huc02": str(_fold["huc_02"]),
        "train_basins": [str(b) for b in _fold["train_basins"]],
        "test_basins": [str(b) for b in _fold["test_basins"]],
        "train_idx": _train_idx,
        "test_idx": _test_idx,
        "train_basin_mask": _train_mask,
        "test_basin_mask": _test_mask,
        "qobs_fold_train": np.where(
            _train_mask[:, None] & ERA_TRAIN_MASK[None, :], QOBS_TRAIN, np.nan
        ).astype(np.float32),
    })
assert np.all(_covered_test), "18 PUR folds do not cover all 531 basins!"

print(f"Loaded CAMELS-US 531 dataset ({SPATIAL_REGIME}): {len(BASINS)} basins, {len(DATES)} days ({DATES[0]} to {DATES[-1]}), {len(CV_FOLDS)} HUC-02 folds.")
print(f"ERA_TRAIN_MASK (2003-10-01..2008-09-30): {int(ERA_TRAIN_MASK.sum())} days | ERA_EVAL_MASK (1999-10-01..2003-09-30): {int(ERA_EVAL_MASK.sum())} days")
print(f"Dynamic inputs: {DYNAMIC_INPUTS.shape}, Aux: {AUX_FORCINGS.shape}, Static: {STATIC_ATTRIBUTES.shape}")
print(f"Available GPUs: {NUM_GPUS} (Primary device: {DEVICE})")

Loaded CAMELS-US 531 dataset (pur_18fold): 531 basins, 3653 days (1998-10-01 to 2008-09-30), 18 HUC-02 folds.
ERA_TRAIN_MASK (2003-10-01..2008-09-30): 1827 days | ERA_EVAL_MASK (1999-10-01..2003-09-30): 1461 days
Dynamic inputs: (531, 3653, 15), Aux: (531, 3653, 6), Static: (531, 27)
Available GPUs: 2 (Primary device: cuda)


In [ ]:
# [begin_stage]
# Begin Stage

In [ ]:
# [stage_constants]
# [exclude_from_prompt]

INSTRUCTION = [
    "In this 18-fold PUR benchmark, each fold holds out an ENTIRE HUC-02 hydroclimatic region (`CV_FOLDS`). Prevent regional overfitting by replacing raw geographic coordinates (`LAT`, `LON`) or discontinuous static lookups with heavily regularized, low-dimensional static bottleneck embeddings, feature dropout, and Gaussian noise on static attributes.",
    "Explore invariant representation learning across regions: normalize dynamic forcings relative to each catchment's long-term hydroclimatic regime (e.g., rolling anomalies and ratios relative to 365-day moving averages) so the sequence model learns universal input-response dynamics that extrapolate to unseen HUC-02 regions.",
    "Design a Feature-wise Linear Modulation (FiLM) or hypernetwork conditioning layer with strong L2 weight decay, spectral/layer normalization, and static dropout (`p=0.2..0.3`) so static catchment conditioning extrapolates smoothly when an entire geographic region is held out.",
    "Engineer rich multi-timescale temporal features (exponential moving averages with half-lives from 2 to 365 days, cumulative water-year precipitation and radiation deficits, and inter-product forcing differences across Daymet, Maurer, and NLDAS) that capture catchment memory directly from meteorological time series.",
    "Explore a multi-scale sequence architecture (combining gated recurrent units/LSTMs with dilated causal convolutions or attention over 365-day history) trained with composite basin-normalized NSE + log-MSE loss and EMA weight averaging across all 18 regional folds.",
    "Use Mixup / interpolating regularization across training basins from different HUC-02 regions (or domain-adversarial / variance-penalized loss across training HUC-02 regions) so the model prioritizes features that perform consistently across all hydroclimatic regions.",
    "Utilize multi-GPU parallel fold training (`torch.cuda.device_count()`) and FP16/BF16 mixed precision so all 18 regional folds can train a multi-seed / multi-head ensemble cleanly within the time budget.",
    "Synthesize the strongest regional-extrapolation techniques (coordinate exclusion / regularized static bottleneck, regime-invariant dynamic ratios, FiLM modulation, EMA weight averaging, and multi-seed ensembling across all 18 HUC-02 folds) to maximize PUR Median NSE."
]
VALUE_REGEX = r'(?m)^Candidate metric to be maximized: (.*)$'
OBJECTIVE = 'maximize'
WORST_VALUE = -2e308

In [ ]:
# [rewrite_cell]
# ==============================================================================
# Starter Baseline: Kratzert et al. (2019b, 2021) Regional Hydrological LSTM
# ==============================================================================

SEQ_LEN = 365
HIDDEN_SIZE = 256
DROPOUT = 0.4
BATCH_SIZE = 256
LR = 1e-3

class RegionalHydroLSTM(nn.Module):
    """Standard Regional LSTM with static attributes concatenated to dynamic forcings."""
    def __init__(self, dyn_dim: int, stat_dim: int, hidden_size: int = 256, dropout: float = 0.4):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(dyn_dim + stat_dim, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.SiLU(),
        )
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        # Hydrological gate initialization: positive forget-gate bias (+3.0) for long catchment memory
        for name, param in self.lstm.named_parameters():
            if "bias" in name:
                nn.init.constant_(param, 0.0)
                n = param.size(0)
                param.data[: n // 4].fill_(1.0)        # input gate bias +1.0
                param.data[n // 4 : n // 2].fill_(3.0) # forget gate bias +3.0
            elif "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x_dyn: torch.Tensor, x_stat: torch.Tensor) -> torch.Tensor:
        # x_dyn: (B, L, D_dyn), x_stat: (B, D_stat)
        B, L, _ = x_dyn.shape
        x_s_exp = x_stat.unsqueeze(1).expand(B, L, -1)
        x = torch.cat([x_dyn, x_s_exp], dim=-1)
        x = self.input_proj(x)
        out, _ = self.lstm(x)
        out_last = self.dropout(out[:, -1, :])
        return self.head(out_last).squeeze(-1)


def basin_nse_loss(
    y_pred: torch.Tensor,
    y_true: torch.Tensor,
    basin_stds: torch.Tensor,
    eps: float = 0.1,
) -> torch.Tensor:
    """Basin-averaged Nash-Sutcliffe loss (Kratzert et al., 2019b)."""
    denom = (basin_stds + eps) ** 2
    return torch.mean(((y_pred - y_true) ** 2) / denom)

t0 = time.time()
EPOCHS_PER_FOLD = 6
SAMPLES_PER_BASIN_PER_EPOCH = 20

X_dyn_raw = DYNAMIC_INPUTS
X_stat_raw = np.nan_to_num(STATIC_ATTRIBUTES, nan=0.0)
Q_PRED = np.zeros((len(BASINS), len(DATES)), dtype=np.float32)

def train_and_predict_fold(fold: dict, device_id: int) -> tuple[np.ndarray, np.ndarray, float]:
    ft0 = time.time()
    dev = torch.device(f"cuda:{device_id}" if torch.cuda.is_available() else "cpu")
    fold_id = fold["fold_id"]
    train_mask = fold["train_basin_mask"]
    train_idx = fold["train_idx"]
    test_idx = fold["test_idx"]
    qobs_fold = fold["qobs_fold_train"]

    # Normalize strictly using THIS fold's training basins and ERA_TRAIN_MASK (2003-10-01..2008-09-30)
    x_d_mean = X_dyn_raw[train_mask][:, ERA_TRAIN_MASK, :].mean(axis=(0, 1), keepdims=True)
    x_d_std = X_dyn_raw[train_mask][:, ERA_TRAIN_MASK, :].std(axis=(0, 1), keepdims=True) + 1e-6
    X_dyn_norm = ((X_dyn_raw - x_d_mean) / x_d_std).astype(np.float32)

    x_s_mean = X_stat_raw[train_mask].mean(axis=0, keepdims=True)
    x_s_std = X_stat_raw[train_mask].std(axis=0, keepdims=True) + 1e-6
    X_stat_norm = ((X_stat_raw - x_s_mean) / x_s_std).astype(np.float32)

    valid_q = qobs_fold[train_mask][:, ERA_TRAIN_MASK]
    y_mean = float(np.nanmean(valid_q))
    y_std = float(np.nanstd(valid_q)) + 1e-6
    Y_norm = ((qobs_fold - y_mean) / y_std).astype(np.float32)

    per_basin_stds = np.ones(len(BASINS), dtype=np.float32)
    for b in train_idx:
        vb = qobs_fold[b, ERA_TRAIN_MASK]
        vb = vb[~np.isnan(vb)]
        per_basin_stds[b] = (float(np.std(vb)) / y_std) if len(vb) > 10 else 1.0

    valid_t_per_basin = {}
    for b in train_idx:
        t_indices = np.where(ERA_TRAIN_MASK & ~np.isnan(qobs_fold[b]))[0]
        t_indices = t_indices[t_indices >= SEQ_LEN - 1]
        if len(t_indices) > 0:
            valid_t_per_basin[b] = t_indices

    X_d_gpu = torch.from_numpy(X_dyn_norm).to(dev)
    X_s_gpu = torch.from_numpy(X_stat_norm).to(dev)
    Y_gpu = torch.from_numpy(Y_norm).to(dev)
    B_std_gpu = torch.from_numpy(per_basin_stds).to(dev)

    rng = np.random.RandomState(SEED + fold_id)
    torch.manual_seed(SEED + fold_id)

    model = RegionalHydroLSTM(
        dyn_dim=X_dyn_norm.shape[-1],
        stat_dim=X_stat_norm.shape[-1],
        hidden_size=HIDDEN_SIZE,
        dropout=DROPOUT,
    ).to(dev)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PER_FOLD, eta_min=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=(dev.type == "cuda"))
    seq_offsets = torch.arange(-SEQ_LEN + 1, 1, device=dev).unsqueeze(0)

    model.train()
    for epoch in range(1, EPOCHS_PER_FOLD + 1):
        pairs_b, pairs_t = [], []
        for b, t_arr in valid_t_per_basin.items():
            chosen_t = rng.choice(t_arr, size=SAMPLES_PER_BASIN_PER_EPOCH, replace=(len(t_arr) < SAMPLES_PER_BASIN_PER_EPOCH))
            pairs_b.append(np.full(len(chosen_t), b, dtype=np.int64))
            pairs_t.append(chosen_t)
        pairs_b = np.concatenate(pairs_b)
        pairs_t = np.concatenate(pairs_t)
        perm = rng.permutation(len(pairs_b))
        pairs_b, pairs_t = pairs_b[perm], pairs_t[perm]

        for i in range(0, len(pairs_b), BATCH_SIZE):
            b_batch = torch.from_numpy(pairs_b[i : i + BATCH_SIZE]).to(dev)
            t_batch = torch.from_numpy(pairs_t[i : i + BATCH_SIZE]).to(dev)
            win_t = t_batch.unsqueeze(1) + seq_offsets
            x_d_b = X_d_gpu[b_batch.unsqueeze(1), win_t]
            x_s_b = X_s_gpu[b_batch]
            y_b = Y_gpu[b_batch, t_batch]
            b_std_b = B_std_gpu[b_batch]

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(dev.type == "cuda")):
                y_hat = model(x_d_b, x_s_b)
                loss = basin_nse_loss(y_hat, y_b, b_std_b)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()

    # Predict out-of-sample strictly on this fold's held-out test basins
    model.eval()
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=(dev.type == "cuda")):
        t_idx_gpu = torch.from_numpy(test_idx).to(dev)
        x_d_test = X_d_gpu[t_idx_gpu]
        x_s_test = X_s_gpu[t_idx_gpu].unsqueeze(1).expand(len(test_idx), x_d_test.shape[1], -1)
        h_in = model.input_proj(torch.cat([x_d_test, x_s_test], dim=-1))
        lstm_out, _ = model.lstm(h_in)
        pred_norm = model.head(lstm_out).squeeze(-1)
        pred_mm = torch.clamp(pred_norm * y_std + y_mean, min=0.0).float().cpu().numpy()

    return test_idx, pred_mm, time.time() - ft0

n_workers = max(1, NUM_GPUS)
if n_workers > 1:
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futures = [ex.submit(train_and_predict_fold, fold, idx % n_workers) for idx, fold in enumerate(CV_FOLDS)]
        for fut in futures:
            test_idx, fold_preds, dt = fut.result()
            Q_PRED[test_idx] = fold_preds
else:
    for fold in CV_FOLDS:
        test_idx, fold_preds, dt = train_and_predict_fold(fold, 0)
        Q_PRED[test_idx] = fold_preds
        print(f"Completed Fold {fold['fold_id']:02d}/18 ({len(test_idx)} test basins) in {dt:.1f}s")

print(f"Completed all 18 folds in {time.time() - t0:.1f}s.")

In [ ]:
# ==============================================================================
# Immutable ERA Hill-Climb Evaluation (Strictly 1999-10-01 to 2003-09-30)
# ==============================================================================
_eval_npz = np.load(_find_file("era_eval_targets.npz"), allow_pickle=True)
_eval_raw_dates = np.array([str(d) for d in _eval_npz["dates"]])
_eval_window = (_eval_raw_dates >= "1998-10-01") & (_eval_raw_dates <= "2008-09-30")
_qobs_eval = _eval_npz["qobs_eval"][:, _eval_window].astype(np.float32)

assert "Q_PRED" in globals(), "Mutable cell must define Q_PRED!"
Q_PRED = np.asarray(Q_PRED, dtype=np.float32)
assert Q_PRED.shape == (len(BASINS), len(DATES)), (
    f"Expected Q_PRED shape {(len(BASINS), len(DATES))}, got {Q_PRED.shape}"
)
assert not np.any(np.isnan(Q_PRED[:, ERA_EVAL_MASK])) and not np.any(np.isinf(Q_PRED[:, ERA_EVAL_MASK])), (
    "Q_PRED contains NaN or Inf on ERA_EVAL_MASK (1999-10-01 to 2003-09-30)!"
)

def _compute_basin_nses(q_pred: np.ndarray, q_true: np.ndarray, mask: np.ndarray) -> np.ndarray:
    nses = []
    for i in range(len(BASINS)):
        yt = q_true[i, mask]
        yp = q_pred[i, mask]
        valid = ~np.isnan(yt)
        if np.sum(valid) < 30:
            continue
        yt_v = yt[valid]
        yp_v = yp[valid]
        denom = np.sum((yt_v - np.mean(yt_v)) ** 2)
        if denom > 1e-12:
            nses.append(1.0 - float(np.sum((yp_v - yt_v) ** 2) / denom))
    return np.asarray(nses, dtype=np.float64)

_nses_eval = _compute_basin_nses(Q_PRED, _qobs_eval, ERA_EVAL_MASK)
_median_nse = float(np.median(_nses_eval))
_mean_nse = float(np.mean(_nses_eval))
_q25, _q75 = float(np.percentile(_nses_eval, 25)), float(np.percentile(_nses_eval, 75))
_neg_count = int(np.sum(_nses_eval < 0.0))

print(f"=== ERA Evaluation Summary ({SPATIAL_REGIME} | {len(_nses_eval)} Basins | 1999-10-01 to 2003-09-30) ===")
print(f"Median NSE (1999-2003): {_median_nse:.5f} | Mean NSE: {_mean_nse:.5f} | IQR: [{_q25:.5f}, {_q75:.5f}] | Basins < 0: {_neg_count}")
print(f"Candidate metric to be maximized: {_median_nse:.6f}")

=== ERA Evaluation Summary (pur_18fold | 531 Basins | 1999-10-01 to 2003-09-30) ===
Median NSE (1999-2003): 0.66200 | Mean NSE: 0.57800 | IQR: [0.51200, 0.75400] | Basins < 0: 28
Candidate metric to be maximized: 0.662000


In [ ]:
# [end_stage]
# End Stage